# Классификация автоматизированного трафика

Цель: предсказать `target=1` для каждой cookie из test по событиям **только** из интервала `[window_start_ts, window_end_ts)`. Основная метрика — официальный Precision при Recall ≥ 0.70.




In [10]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from catboost import CatBoostClassifier
import catboost
from sklearn.metrics import roc_auc_score, average_precision_score
SEED = 42
print('Python:', sys.version.split()[0], '| numpy:', np.__version__, '| pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__, 'Версия CatBoost:', catboost.__version__)

def path_for(name):
    options = [Path('data') / name, Path(name)]
    for path in options:
        if path.is_file():
            return path
    raise FileNotFoundError(f'Не найден {name}. Положите файл в data/ или рядом с ноутбуком.')


Python: 3.12.4 | numpy: 2.5.2 | pandas: 3.0.5 | scikit-learn: 1.9.0 Версия CatBoost: 1.2.10


## 1. Загрузка и проверка данных

При чтении преобразуем временные столбцы в даты. Затем проверяем
уникальность `cookie_id`, корректность границ окон и наличие меток в train.
Отдельно смотрим на частоты типов событий, пропуски и полные дубликаты:
это помогает заметить особенности данных до построения признаков.

In [11]:
metric_path = path_for('metric.py')
sys.path.insert(0, str(metric_path.parent.resolve()))
from metric import precision_at_recall

TIME_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv(path_for('train.csv'), parse_dates=TIME_COLS)
test = pd.read_csv(path_for('test.csv'), parse_dates=TIME_COLS)
events = pd.read_csv(path_for('events.csv.gz'), parse_dates=['event_ts'])
assert train.cookie_id.is_unique and test.cookie_id.is_unique
assert not set(train.cookie_id) & set(test.cookie_id)
assert train.target.notna().all() and set(train.target.unique()) <= {0, 1}
assert (train.window_end_ts > train.window_start_ts).all()
assert (test.window_end_ts > test.window_start_ts).all()
print('Размеры:', train.shape, test.shape, events.shape)
print('Доля класса 1:', round(train.target.mean(), 4))
print('Окна train:', train.window_start_ts.min(), train.window_start_ts.max())
print('Окна test:', test.window_start_ts.min(), test.window_start_ts.max())
print('Типы событий:', events.event_name.value_counts(dropna=False).head(20).to_dict())
print('Доли пропусков:', events.isna().mean().round(3).to_dict())
print('Полные дубликаты событий:', events.duplicated().sum())


Размеры: (11091, 5) (4909, 4) (328905, 14)
Доля класса 1: 0.0811
Окна train: 2026-04-06 00:00:00 2026-04-19 00:00:00
Окна test: 2026-04-20 00:00:00 2026-04-26 00:00:00
Типы событий: {'item_view': 120817, 'search_results_view': 100402, 'photo_swipe': 36517, 'favorite_add': 19049, 'seller_page_view': 17403, 'contact_phone_show': 11316, 'captcha_shown': 7928, 'login': 6267, 'contact_chat_open': 6125, 'contact_message_sent': 3081}
Доли пропусков: {'cookie_id': 0.0, 'event_ts': 0.0, 'eid': 0.0, 'event_name': 0.0, 'platform': 0.0, 'user_agent': 0.0, 'item_id': 0.348, 'item_category': 0.1, 'item_location': 0.072, 'seller_type': 0.407, 'search_query': 0.695, 'search_page': 0.695, 'pointer_x': 0.67, 'pointer_y': 0.67}
Полные дубликаты событий: 4863


## 2. Формирование признаков

Единица предсказания - кука. Соединяем события с метаданными без метки, затем строго отсекаем всё за пределами окна. Полные дубликаты не удаляем молча: они могут отражать реальные одинаковые события; их число проверено выше. Пропущенные поля, допустимые для отдельных типов событий, учитываются через число заполненных значений. Сырые `cookie_id`, `user_agent` и строки поисковых запросов не передаются модели; используются только агрегаты.

Признаки: объём событий; количество и разнообразие действий/объявлений/поиска/платформ; интервалы и плотность событий; возраст cookie на момент конца окна; координаты указателя и глубина поиска. Число дней от начала календаря и сам идентификатор не используются. Признаки времени появления cookie опираются только на данные, доступные к концу окна.


In [12]:
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='inner', validate='many_to_one')
ev = ev.loc[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].copy()
ev.sort_values(['cookie_id', 'event_ts'], kind='stable', inplace=True)
assert ((ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)).all()
print('Событий в разрешённых окнах:', len(ev))


Событий в разрешённых окнах: 288126


In [13]:
def build_features(meta, ev):
    idx = pd.Index(meta.cookie_id, name='cookie_id')
    f = pd.DataFrame(index=idx)
    f['cookie_age_days'] = ((meta.window_end_ts - meta.cookie_created_at).dt.total_seconds() / 86400).to_numpy()
    f['start_weekday'] = meta.window_start_ts.dt.dayofweek.to_numpy()
    f['n_events'] = ev.groupby('cookie_id').size()
    distinct = ['item_id', 'item_category', 'item_location', 'seller_type', 'search_query', 'search_page', 'event_name', 'platform', 'user_agent', 'eid']
    for col in distinct:
        if col in ev.columns:
            g = ev.groupby('cookie_id')[col]
            f[col + '_nunique'] = g.nunique()
            f[col + '_nonmissing'] = g.count()
    if 'item_id' in ev.columns:
        item_counts = ev.dropna(subset=['item_id']).groupby(['cookie_id', 'item_id']).size()
        f['max_item_repeats'] = item_counts.groupby(level=0).max()
        f['mean_item_repeats'] = item_counts.groupby(level=0).mean()
    if 'event_name' in ev.columns:
        counts = pd.crosstab(ev.cookie_id, ev.event_name.fillna('_missing_').astype(str)).add_prefix('event_count_')
        f = f.join(counts)
    if 'platform' in ev.columns:
        platform = pd.crosstab(ev.cookie_id, ev.platform.fillna('_missing_').astype(str)).add_prefix('platform_count_')
        f = f.join(platform)
    if 'search_page' in ev.columns:
        page = pd.to_numeric(ev.search_page, errors='coerce')
        f['search_page_max'] = page.groupby(ev.cookie_id).max()
        f['search_page_mean'] = page.groupby(ev.cookie_id).mean()
    for col in ['pointer_x', 'pointer_y']:
        if col in ev.columns:
            pointer = pd.to_numeric(ev[col], errors='coerce')
            f[col + '_std'] = pointer.groupby(ev.cookie_id).std()
            f[col + '_nunique'] = pointer.groupby(ev.cookie_id).nunique()
    seconds = (ev.event_ts - ev.window_start_ts).dt.total_seconds()
    g = seconds.groupby(ev.cookie_id)
    f['first_event_hour'] = g.min() / 3600
    f['last_event_hour'] = g.max() / 3600
    f['active_span_hours'] = (g.max() - g.min()) / 3600
    f['active_hours'] = ev.event_ts.dt.floor('h').groupby(ev.cookie_id).nunique()
    gaps = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    gap_group = gaps.groupby(ev.cookie_id)
    for name, series in [('gap_mean', gap_group.mean()), ('gap_std', gap_group.std()), ('gap_median', gap_group.median()), ('gap_min', gap_group.min())]:
        f[name] = series
    for threshold in [1, 5, 30, 60]:
        f[f'gaps_le_{threshold}'] = gaps.le(threshold).groupby(ev.cookie_id).sum()
    f['first_hour_events'] = seconds.lt(3600).groupby(ev.cookie_id).sum()
    f['last_hour_events'] = seconds.ge(23 * 3600).groupby(ev.cookie_id).sum()
    f = f.replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
    assert f.index.is_unique and len(f) == len(meta)
    return f

features = build_features(meta, ev)
Xtr, Xte = features.iloc[:len(train)].copy(), features.iloc[len(train):].copy()
ytr = train.target.to_numpy()
assert Xtr.index.tolist() == train.cookie_id.tolist()
assert Xte.index.tolist() == test.cookie_id.tolist()
print('Признаков:', Xtr.shape[1], '| train:', Xtr.shape, '| test:', Xte.shape)


Признаков: 65 | train: (11091, 65) | test: (4909, 65)


## 3. Временная валидация и baseline

Последние три дня train (начиная с 17 апреля 2026) — валидация, предыдущие дни — обучение. Test лежит позже. Сначала сравниваем константу и baseline из quickstart: случайный лес с количеством событий и уникальных объявлений. Все результаты ниже рассчитываются при запуске; PR-AUC и ROC-AUC диагностические, выбор модели делается по официальной метрике.


In [14]:
valid = train.window_start_ts.ge(pd.Timestamp('2026-04-17')).to_numpy()
assert valid.any() and (~valid).any()
assert set(ytr[valid]) == {0, 1} and set(ytr[~valid]) == {0, 1}
print('Train/validation:', (~valid).sum(), valid.sum(), 'prevalence:', ytr[~valid].mean(), ytr[valid].mean())
base_cols = ['n_events', 'item_id_nunique']
assert set(base_cols).issubset(Xtr.columns)
constant = np.full(valid.sum(), ytr[~valid].mean())
base_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
base_model.fit(Xtr.loc[~valid, base_cols], ytr[~valid])
base_pred = base_model.predict_proba(Xtr.loc[valid, base_cols])[:, 1]

def report(name, pred):
    return {'name': name, 'P@R>=0.70': precision_at_recall(ytr[valid], pred),
            'PR-AUC': average_precision_score(ytr[valid], pred),
            'ROC-AUC': roc_auc_score(ytr[valid], pred)}

results = [report('constant', constant), report('quickstart_baseline', base_pred)]
pd.DataFrame(results)


Train/validation: 9140 1951 prevalence: 0.08085339168490153 0.08200922603792926


,name,P@R>=0.70,PR-AUC,ROC-AUC
0,constant,0.082009,0.082009,0.500000
1,quickstart_baseline,0.106161,0.180385,0.608129


## 4. Улучшенные модели и выбор

Сравниваем три модели на одном наборе построенных признаков: Random Forest, HistGradientBoosting и CatBoost. Не подбираем много конфигураций по трём дням, чтобы не подогнать модель к валидации. При равном результате остаётся более простой baseline. После выбора модель переобучается на **всём** train.


In [15]:
candidates = [
    (
        'random_forest_behavior',
        RandomForestClassifier(
            n_estimators=450,
            min_samples_leaf=3,
            max_features=0.7, 
            random_state=SEED, 
            n_jobs=-1)
    ),
    (
        'hist_gradient_boosting',
        HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.04, 
            max_leaf_nodes=15, 
            min_samples_leaf=25, 
            l2_regularization=3, 
            random_state=SEED)
    ),
    (
        'catboost',
        CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.04,
            l2_leaf_reg=3,
            loss_function='Logloss',
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False)
    )
]
for name, model in candidates:
    model.fit(Xtr.iloc[~valid], ytr[~valid])
    pred = model.predict_proba(Xtr.iloc[valid])[:, 1]
    results.append(report(name, pred))
comparison = pd.DataFrame(results).sort_values('P@R>=0.70', ascending=False, kind='stable')
print(comparison.to_string(index=False))
selected = max(results[1:], key=lambda row: row['P@R>=0.70'])
print('Выбрана модель:', selected['name'])


                  name  P@R>=0.70   PR-AUC  ROC-AUC
              catboost   0.722581 0.758044 0.928901
hist_gradient_boosting   0.715190 0.754085 0.927192
random_forest_behavior   0.666667 0.721978 0.906002
   quickstart_baseline   0.106161 0.180385 0.608129
              constant   0.082009 0.082009 0.500000
Выбрана модель: catboost


## 5. Обучение и отправка

Формируем ровно одну строку на cookie из test; никаких порогов и тестовых меток не используем. Дополнительно проверяем схему и идентификаторы.


In [17]:
final_model = next(
    model for name, model in candidates 
    if name == selected['name']
)
final_model.fit(Xtr, ytr)
submission = pd.DataFrame({
    'cookie_id': test.cookie_id.to_numpy(), 
    'score': final_model.predict_proba(Xte)[:, 1]
})
assert list(submission.columns) == ['cookie_id', 'score']
assert len(submission) == len(test) and submission.cookie_id.is_unique
assert submission.cookie_id.equals(test.cookie_id)
assert submission.score.notna().all() and submission.score.between(0, 1).all()
submission.to_csv('submission.csv', index=False)
print(
    'submission.csv сохранён:',
    submission.shape, 
    '| score min/max:', 
    submission.score.min(), 
    submission.score.max()
)

submission.head()

submission.csv сохранён: (4909, 2) | score min/max: 0.0002305914417710762 0.9992857844293408


,cookie_id,score
0,ck_315fb710a0e371e7,0.000331
1,ck_a76ee3b3e3e522fd,0.084606
2,ck_94c9a4d382689e82,0.058396
3,ck_8eaf9509ad9462a0,0.001147
4,ck_9a88a5a989cb5bc6,0.010647


## Подход к решению

Задача - для каждой `cookie_id` из `test.csv` получить `score` от 0 до 1:
чем выше значение, тем больше оснований отнести её к трафику
автоматизированного сервиса сбора данных (`target = 1`).

В `train.csv` и `test.csv` находятся метаданные cookie и границы
суточного окна наблюдения, а в `events.csv.gz` - отдельные события.
Сначала я связываю события с cookie по `cookie_id` и оставляю только
события внутри разрешённого окна:
`window_start_ts <= event_ts < window_end_ts`.
Затем сортирую их по времени внутри каждой cookie.

Из истории событий я формирую одну строку числовых признаков на cookie.
Признаки описывают объём активности, частоты типов действий, разнообразие
объявлений, категорий и поисковых запросов, повторные обращения к
объявлениям, страницы поиска и временной ритм действий - например,
промежутки между соседними событиями. Дополнительно использую возраст
cookie, день недели, агрегаты по платформе, User-Agent и координатам
указателя. Сырые значения `cookie_id`, `user_agent` и текст запросов
не передаются модели как признаки.

Идея выбора поведенческих признаков состоит в том, что одинаковое число
событий может соответствовать разным сценариям работы: важны также
состав действий, разнообразие просмотренного и распределение событий
по времени. Эти признаки рассматриваются как гипотезы; сам по себе
отдельный короткий интервал или частый просмотр объявления не доказывает,
что cookie относится к классу 1.

Для проверки качества я использую временное разделение `train.csv`:
обучаю модели на более ранних окнах, а строки с `window_start_ts`
начиная с 17 апреля 2026 года оставляю для валидации. Ответы этой
отложенной части не передаются модели при обучении. Сравниваю константный
прогноз, baseline из quickstart с двумя признаками и три модели на
расширенном наборе признаков: Random Forest, HistGradientBoosting
и CatBoost. Основная метрика - `Precision при Recall ≥ 70%`,
рассчитанная официальной функцией из `metric.py`.

На временной валидации получены результаты: константа - `0,082009`,
baseline - `0,106161`, Random Forest - `0,666667`,
HistGradientBoosting - `0,715190`, CatBoost - `0,722581`.
Лучший из проверенных вариантов - CatBoost. После выбора модель
обучается заново на всём `train.csv` и выдаёт `score` для `test.csv`;
результат сохраняется в `submission.csv`.